# Explorer — Nemotron Post-Training v3 EDA

Deep exploratory data analysis of the **Nemotron post-training v3** collection processed through the hfdata pipeline.

Datasets: Instruction-Following-Chat · Math-Proofs · Math-v2 · Science · SWE · Competitive-Programming · Agentic · RL-Training-Blend

## 0. Setup & Load

In [ ]:
import json
import os
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

figure_dir = Path("figures/nemotron-post-training-v3")
figure_dir.mkdir(parents=True, exist_ok=True)

latex_template = go.layout.Template()
latex_template.layout = go.Layout(
    paper_bgcolor="white",
    plot_bgcolor="white",
    font=dict(family="Computer Modern, CMU Serif, STIX Two Text, Latin Modern Roman, serif"),
    xaxis=dict(showgrid=True, gridcolor="lightgray", gridwidth=1, zeroline=False),
    yaxis=dict(showgrid=True, gridcolor="lightgray", gridwidth=1, zeroline=False),
    coloraxis=dict(colorscale="piyg", colorbar=dict(outlinewidth=0)),
    colorscale=dict(sequential="piyg", sequentialminus="piyg", diverging="piyg"),
)
pio.templates["latex_white"] = latex_template
pio.templates.default = "plotly_white+latex_white"

COLLECTION_ROOT = Path("datasets/nvidia")
MODEL_ID = "nvidia/llama-embed-nemotron-8b"

DATASETS = [
    "Nemotron-Instruction-Following-Chat-v1",
    "Nemotron-Math-Proofs-v1",
    "Nemotron-Math-v2",
    "Nemotron-Science-v1",
    "Nemotron-SWE-v1",
    "Nemotron-Competitive-Programming-v1",
    "Nemotron-Agentic-v1",
    "Nemotron-3-Nano-RL-Training-Blend",
]

# ── Discover and load all preprocessed splits ────────────────────────────────
frames, red_frames = [], []
for ds_name in DATASETS:
    ds_root = COLLECTION_ROOT / ds_name / "default"
    if not ds_root.exists():
        print(f"  SKIP {ds_name}: directory not found")
        continue
    for split_dir in sorted(ds_root.iterdir()):
        if not split_dir.is_dir():
            continue
        pp_dir = split_dir / "preprocessed"
        pq_files = sorted(pp_dir.glob("*.parquet")) if pp_dir.exists() else []
        if not pq_files:
            continue
        split_df = pd.concat([pd.read_parquet(f) for f in pq_files], ignore_index=True)
        split_df["dataset"] = ds_name
        split_df["split"] = split_dir.name
        frames.append(split_df)
        # Reduced embeddings
        red_dir = split_dir / "embreduced" / MODEL_ID.replace("/", "/")
        red_pq = sorted(red_dir.glob("*.parquet")) if red_dir.exists() else []
        if red_pq:
            split_red = pd.concat([pd.read_parquet(f) for f in red_pq], ignore_index=True)
            split_red["dataset"] = ds_name
            split_red["split"] = split_dir.name
            red_frames.append(split_red)

df = pd.concat(frames, ignore_index=True)
red_df = pd.concat(red_frames, ignore_index=True) if red_frames else pd.DataFrame()

# Expand metadata_json into flat columns
meta_base = df.drop(columns=["metadata_json"]).reset_index(drop=True)
meta_expanded = pd.json_normalize(df["metadata_json"].apply(json.loads))
meta_expanded = meta_expanded.drop(
    columns=[c for c in meta_expanded.columns if c in meta_base.columns], errors="ignore"
)
df = pd.concat([meta_base, meta_expanded], axis=1)

print(f"Total rows       : {len(df):,}")
print(f"Reduced rows     : {len(red_df):,}")
print(f"Datasets loaded  : {df['dataset'].nunique()}")
print(f"Splits loaded    : {df[['dataset','split']].drop_duplicates().shape[0]}")
print(f"Columns (meta)   : {list(df.columns)}")
print(f"Columns (reduced): {list(red_df.columns)}")

## 1. Metadata Schema Study

In [ ]:
schema_info = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "non_null": df.notnull().sum(),
    "null_pct": (df.isnull().sum() / len(df) * 100).round(1),
    "unique": df.nunique(),
    "sample": df.iloc[0].astype(str).str[:80],
})
schema_info

In [ ]:
coverage = df.groupby("dataset").apply(
    lambda g: g.notnull().mean().round(3)
).T
coverage.style.background_gradient(cmap="YlGn", axis=1).format("{:.1%}")

## 2. Disk Usage & Text Length

In [ ]:
def dir_size_mb(path: Path) -> float:
    return sum(f.stat().st_size for f in path.rglob("*") if f.is_file()) / 1e6

sizes = {}
for ds_name in DATASETS:
    ds_root = COLLECTION_ROOT / ds_name / "default"
    if not ds_root.exists():
        continue
    for split_dir in sorted(ds_root.iterdir()):
        if not split_dir.is_dir():
            continue
        for stage in ["raw", "preprocessed"]:
            p = split_dir / stage
            if p.exists() and any(p.rglob("*.parquet")):
                sizes[f"{ds_name}/{split_dir.name}/{stage}"] = dir_size_mb(p)
        for sub in ["embeddings", "embreduced"]:
            p = split_dir / sub / MODEL_ID.replace("/", "/")
            if p.exists() and any(p.rglob("*.parquet")):
                sizes[f"{ds_name}/{split_dir.name}/{sub}"] = dir_size_mb(p)

print(f"Total records: {len(df):,}\n")
print("Disk usage (MB):")
for k, v in sorted(sizes.items()):
    print(f"  {k:65s} {v:>10,.1f} MB")
print(f"  {'TOTAL':65s} {sum(sizes.values()):>10,.1f} MB")

In [ ]:
df["text_len"] = df["text"].str.len()

fig = px.histogram(
    df, x="text_len", nbins=100, color="dataset",
    title="Text Length Distribution (all datasets)",
    labels={"text_len": "Character count", "dataset": "Dataset"},
    marginal="box",
)
fig.update_layout(height=600, barmode="overlay")
fig.update_traces(opacity=0.6)
fig.show()
fig.write_image(figure_dir / "Text Length Distribution.png", scale=2)

print("\nText length stats by dataset:")
print(df.groupby("dataset")["text_len"].describe().round(0).to_string())

## 3. Category Distributions

In [ ]:
# ── Rows per dataset ──────────────────────────────────────────────────────────
ds_counts = df["dataset"].value_counts()
fig = px.bar(
    x=ds_counts.values, y=ds_counts.index, orientation="h",
    title="Rows per Dataset",
    labels={"x": "Count", "y": "Dataset"},
    color=ds_counts.index,
)
fig.update_layout(height=500, yaxis=dict(autorange="reversed"), showlegend=False)
fig.show()
fig.write_image(figure_dir / "Rows per Dataset.png", scale=2)

In [ ]:
# ── Rows per split (within each dataset) ─────────────────────────────────────
split_counts = df.groupby(["dataset", "split"]).size().reset_index(name="count")
fig = px.bar(
    split_counts, x="count", y="split", color="dataset", orientation="h",
    title="Rows per Split",
    labels={"count": "Count", "split": "Split"},
)
fig.update_layout(height=max(400, len(split_counts) * 30), yaxis=dict(autorange="reversed"))
fig.show()
fig.write_image(figure_dir / "Rows per Split.png", scale=2)

In [ ]:
# ── License distribution ──────────────────────────────────────────────────────
if "license" in df.columns:
    lic_counts = df["license"].value_counts().head(20)
    fig = px.bar(
        x=lic_counts.values, y=lic_counts.index, orientation="h",
        title="License Distribution",
        labels={"x": "Count", "y": "License"},
    )
    fig.update_layout(height=500, yaxis=dict(autorange="reversed"))
    fig.show()
    fig.write_image(figure_dir / "License Distribution.png", scale=2)
else:
    print("No 'license' column found.")

In [ ]:
# ── used_in distribution ──────────────────────────────────────────────────────
if "used_in" in df.columns:
    used_str = df["used_in"].dropna().astype(str)
    used_counts = used_str.value_counts().head(20)
    fig = px.bar(
        x=used_counts.values, y=used_counts.index, orientation="h",
        title="Top 20 'used_in' Values",
        labels={"x": "Count", "y": "used_in"},
    )
    fig.update_layout(height=600, yaxis=dict(autorange="reversed"))
    fig.show()
    fig.write_image(figure_dir / "Top 20 used_in Values.png", scale=2)
else:
    print("No 'used_in' column found.")

In [ ]:
# ── tools distribution ────────────────────────────────────────────────────────
if "tools" in df.columns:
    tools_str = df["tools"].dropna().astype(str)
    tools_counts = tools_str.value_counts().head(20)
    fig = px.bar(
        x=tools_counts.values, y=tools_counts.index, orientation="h",
        title="Top 20 'tools' Values",
        labels={"x": "Count", "y": "tools"},
    )
    fig.update_layout(height=600, yaxis=dict(autorange="reversed"))
    fig.show()
    fig.write_image(figure_dir / "Top 20 tools Values.png", scale=2)
else:
    print("No 'tools' column found.")

In [ ]:
# ── Cross-tab heatmap: dataset x split ────────────────────────────────────────
ct = pd.crosstab(df["dataset"], df["split"])
ct = ct.loc[:, ct.sum() > 0]

fig = px.imshow(
    ct, aspect="auto",
    title="Dataset × Split Heatmap",
    labels=dict(x="Split", y="Dataset", color="Count"),
    color_continuous_scale="BuPu",
    text_auto=True,
)
fig.update_layout(height=500)
fig.show()
fig.write_image(figure_dir / "Dataset x Split Heatmap.png", scale=2)

## 4. 2D Embedding Scatter (Interactive)

Toggle **algorithm** (PCA / t-SNE / UMAP) and **color** (dataset / split) via dropdown.

In [ ]:
# Merge metadata with reduced embeddings (row-aligned per dataset+split)
viz_frames = []
for (ds, sp), grp in df.groupby(["dataset", "split"]):
    red_grp = red_df[(red_df["dataset"] == ds) & (red_df["split"] == sp)]
    if red_grp.empty:
        continue
    n = min(len(grp), len(red_grp))
    chunk = grp.iloc[:n].reset_index(drop=True).copy()
    for col in ["pca_2d_x", "pca_2d_y", "tsne_2d_x", "tsne_2d_y", "umap_2d_x", "umap_2d_y"]:
        if col in red_grp.columns:
            chunk[col] = red_grp[col].values[:n]
    viz_frames.append(chunk)

viz = pd.concat(viz_frames, ignore_index=True) if viz_frames else pd.DataFrame()
print(f"Rows with embeddings: {len(viz):,}")

# ── Subsample for performance ────────────────────────────────────────────────
SAMPLE_N = 80_000
if len(viz) > SAMPLE_N:
    viz_sample = viz.sample(n=SAMPLE_N, random_state=42).reset_index(drop=True)
    print(f"Subsampled to {SAMPLE_N:,} points for interactive plots")
else:
    viz_sample = viz

ALGORITHMS = {
    "PCA":   ("pca_2d_x",  "pca_2d_y"),
    "t-SNE": ("tsne_2d_x", "tsne_2d_y"),
    "UMAP":  ("umap_2d_x", "umap_2d_y"),
}

COLOR_FIELDS = {
    "Dataset": "dataset",
    "Split":   "split",
}

fig = go.Figure()

combo_map = {}
idx = 0
for algo_name, (xcol, ycol) in ALGORITHMS.items():
    for color_name, color_col in COLOR_FIELDS.items():
        for cat in sorted(viz_sample[color_col].unique()):
            mask = viz_sample[color_col] == cat
            fig.add_trace(go.Scattergl(
                x=viz_sample.loc[mask, xcol],
                y=viz_sample.loc[mask, ycol],
                mode="markers",
                marker=dict(size=2, opacity=0.6),
                name=str(cat),
                visible=False,
                text=viz_sample.loc[mask, "text"].str[:80],
                hovertemplate="%{text}<extra>%{fullData.name}</extra>",
            ))
            combo_map.setdefault((algo_name, color_name), []).append(idx)
            idx += 1

default_key = ("PCA", "Dataset")
for i in combo_map.get(default_key, []):
    fig.data[i].visible = True

algo_list = list(ALGORITHMS.keys())
color_list = list(COLOR_FIELDS.keys())

buttons_combined = []
for algo_name in algo_list:
    for color_name in color_list:
        visible = [False] * len(fig.data)
        for i in combo_map.get((algo_name, color_name), []):
            visible[i] = True
        buttons_combined.append(dict(
            label=f"{algo_name}  · {color_name}",
            method="update",
            args=[{"visible": visible}, {"title": f"2D Embeddings — {algo_name} · {color_name}"}],
        ))

fig.update_layout(
    title="2D Embeddings — PCA · Dataset",
    height=700,
    margin=dict(l=60, r=300, t=80, b=60),
    width=1100,
    xaxis=dict(domain=[0, 0.72], automargin=False),
    yaxis=dict(domain=[0, 1], automargin=False),
    updatemenus=[
        dict(
            buttons=buttons_combined,
            direction="down",
            x=1.0, xanchor="left", y=1.0, yanchor="top",
            showactive=True,
            active=0,
        ),
    ],
    annotations=[
        dict(text="View:", x=1.0, xref="paper", y=1.06, yref="paper",
             showarrow=False, xanchor="left"),
    ],
    legend=dict(itemsizing="constant", x=0.74, xanchor="left", y=0.85, yanchor="top"),
)
fig.show()

for _algo in algo_list:
    for _color in color_list:
        _vis = [False] * len(fig.data)
        for _i in combo_map.get((_algo, _color), []):
            _vis[_i] = True
        for _j, _v in enumerate(_vis):
            fig.data[_j].visible = _v
        fig.layout.title.text = f"2D Embeddings - {_algo} - {_color}"
        fig.write_image(figure_dir / f"2D Embeddings - {_algo} - {_color}.png", scale=2)